# Enriched annotation statistics

Corpus-level numbers for the enriched Accusation and Identity Declaration layers,
as reported in the Data and Models chapter.

The aggregation logic lives in `src/pt_annotation/`, so that every figure in the
thesis regenerates from the stored annotations rather than from state held in this
notebook:

- `identity_claim_role_distribution.py` — claimed-role distribution
- `accusation_distribution.py` — accusation subtypes and counts
- `accusation_target_by_role.py` — accusations received, by the target's true role

Both take `--latex` to emit the table body directly.


In [ ]:
import json
import subprocess
import sys
from collections import Counter
from pathlib import Path

REPO = Path(r"C:\Users\annab\Documents\GitHub\masters_thesis_sdg")
PT = REPO / "src" / "pt_annotation"
if str(PT) not in sys.path:
    sys.path.insert(0, str(PT))

from accusation_target_by_role import load_truth, normalise_name       # noqa: E402
from identity_claim_role_distribution import normalise_role            # noqa: E402


def run_script(name, *args):
    """Run one of the aggregation scripts and print its output."""
    print(subprocess.run([sys.executable, str(PT / name), *args],
                         capture_output=True, text=True).stdout)


## Were Werewolf claims true?

The distribution table records that 162 role mentions claim the Werewolf. This
checks those claims against the ground-truth role assignments: of the players who
said they were a Werewolf, how many actually held a Werewolf card, at the start of
the night and at the end of it.

The identity-claim JSONs do not store the speaker, only the line number, so the
speaker is recovered from the corresponding transcript line.


In [ ]:
IC_ROOT = REPO / "data/processed/lai2023/identity_claim_transcripts/ic_targets"
IC_TXT = REPO / "data/processed/lai2023/identity_claim_transcripts/ready_for_annotation"
IC_MARKER = "<<IDENTITY_CLAIM_TO_RESOLVE>>"


def speakers_for(txt_path):
    """{line_number: speaker} parsed from '[N] Speaker: text' transcript lines."""
    out = {}
    for raw in txt_path.read_text(encoding="utf-8").splitlines():
        if not raw.startswith("["):
            continue
        end = raw.find("]")
        if end == -1 or not raw[1:end].isdigit():
            continue
        rest = raw[end + 1:].replace(IC_MARKER, "").strip()
        if ":" in rest:
            out[int(raw[1:end])] = rest.split(":", 1)[0].strip()
    return out


def werewolf_claim_outcomes():
    truth = load_truth()
    claims = joined = unmatched = 0
    outcome, start_roles = Counter(), Counter()

    for path in sorted(IC_ROOT.rglob("*.json")):
        record = json.loads(path.read_text(encoding="utf-8"))
        meta = record.get("metadata", {})
        key = (meta.get("source"), meta.get("session"), meta.get("game"))
        txt = IC_TXT / (meta.get("source_file") or "").replace("\\", "/")
        speakers = speakers_for(txt) if txt.exists() else {}

        for item in record.get("items", []):
            roles = {normalise_role(r) for r in (item.get("claimed_roles") or [])}
            if "Werewolf" not in roles:
                continue
            claims += 1

            speaker = speakers.get(item.get("line_number"))
            roster = {normalise_name(n): n for n in truth.get(key, {})}
            who = roster.get(normalise_name(speaker)) if speaker else None
            if who is None:
                unmatched += 1
                continue

            joined += 1
            start = (truth[key][who].get("start") or "").strip()
            end = (truth[key][who].get("end") or "").strip()
            start_roles[start] += 1
            outcome[(start.casefold() == "werewolf", end.casefold() == "werewolf")] += 1

    return claims, joined, unmatched, outcome, start_roles


claims, joined, unmatched, outcome, start_roles = werewolf_claim_outcomes()

LABELS = {
    (True, True): "held Werewolf at start and end",
    (True, False): "started Werewolf, lost it overnight",
    (False, True): "acquired Werewolf overnight",
    (False, False): "never held a Werewolf card",
}

print(f"utterances claiming Werewolf : {claims}  (joined {joined}, unmatched {unmatched})")
print()
for combo in [(True, True), (True, False), (False, True), (False, False)]:
    n = outcome[combo]
    print(f"  {LABELS[combo]:<36}{n:>4}  ({100 * n / joined:5.1f}%)")

end_ww = outcome[(True, True)] + outcome[(False, True)]
start_ww = outcome[(True, True)] + outcome[(True, False)]
ever = end_ww + outcome[(True, False)]
print()
print(f"  held a Werewolf card at the END     {end_ww:>4}  ({100 * end_ww / joined:.1f}%)")
print(f"  held one at the START               {start_ww:>4}  ({100 * start_ww / joined:.1f}%)")
print(f"  held one at either point            {ever:>4}  ({100 * ever / joined:.1f}%)")
print()
print("starting role of the claimant:")
for role, n in start_roles.most_common():
    print(f"  {role:<16}{n:>4}  ({100 * n / joined:.1f}%)")


## Distribution tables

The three aggregation scripts, for regenerating the chapter's numbers and table
bodies. Add `--latex` to any of them to get the table body instead of the summary.


In [ ]:
run_script("identity_claim_role_distribution.py")
run_script("accusation_distribution.py")
run_script("accusation_target_by_role.py", "--role-frame", "start")
run_script("accusation_target_by_role.py", "--role-frame", "end")
